In [ ]:
####### fyp2

### Feature Preprocessing (Resize + Pad/Truncate to 768)

This prepares your acoustic and visual features properly for transformer-based multimodal models.

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

#### Load CSV and Inspect Structure

In [2]:
import os
import numpy as np
import pandas as pd

BASE_PATH = "/content/drive/MyDrive/dvlog-dataset"

df = pd.read_csv(os.path.join(BASE_PATH, "labels.csv"))


#### Check of File Availability

In [3]:
missing = []

for idx in df["index"].astype(str):
    folder = os.path.join(BASE_PATH, idx)

    acoustic_file = os.path.join(folder, f"{idx}_acoustic.npy")
    visual_file   = os.path.join(folder, f"{idx}_visual.npy")

    if not (os.path.exists(acoustic_file) and os.path.exists(visual_file)):
        missing.append(idx)

print("Missing files:", missing)
print("Total missing:", len(missing))


Missing files: []
Total missing: 0


#### Loading data

In [4]:
# !pip install tqdm
from tqdm import tqdm

acoustic_list = []
visual_list = []
labels_list = []
folds_list = []
indexes_list = []

for i, row in tqdm(df.iterrows(), total=len(df), desc="Loading .npy files"):
    idx = str(row["index"])

    folder = os.path.join(BASE_PATH, idx)
    acoustic_path = os.path.join(folder, f"{idx}_acoustic.npy")
    visual_path   = os.path.join(folder, f"{idx}_visual.npy")

    if not os.path.exists(acoustic_path) or not os.path.exists(visual_path):
        continue

    # Much faster load
    acoustic = np.load(acoustic_path, mmap_mode="r")
    visual   = np.load(visual_path, mmap_mode="r")

    acoustic_list.append(acoustic)
    visual_list.append(visual)
    labels_list.append(row["label"])
    folds_list.append(row["fold"])
    indexes_list.append(idx)

print("Loaded acoustic:", len(acoustic_list))
print("Loaded visual:", len(visual_list))


Loading .npy files: 100%|██████████| 961/961 [22:53<00:00,  1.43s/it]

Loaded acoustic: 961
Loaded visual: 961


#### Convert to Arrays

In [6]:
acoustic_list = np.array(acoustic_list, dtype=object)
visual_list = np.array(visual_list, dtype=object)
labels_list = np.array(labels_list)
folds_list = np.array(folds_list)
indexes_list = np.array(indexes_list)

print(acoustic_list.shape, visual_list.shape)


(961,) (961,)


### What This Step Does

This preprocessing step prepares your dataset for multimodal transformers by normalizing all feature sequences to a consistent length of 768 using padding for shorter sequences and trimming for longer ones. This ensures compatibility with batch-based transformer models such as MMBT, MulT, Cross-Modal Transformers, and AV-Fusion Transformers. The code also loads the features and labels, organizes them into NumPy arrays, and prints their final shapes for verification.

#### Pad Acoustic & Visual Sequences to Length 768

In [7]:
# STEP 2 — Increase Sequence Length to 768

SEQ_LEN = 768   # longer temporal window → better accuracy

padded_acoustic = []
padded_visual = []
labels = []
folds = []
indexes = []

def pad_sequence(arr, max_len=SEQ_LEN):
    """Pads or truncates input (T, F) to (max_len, F)."""
    T, F = arr.shape

    # Truncate
    if T >= max_len:
        return arr[:max_len]

    # Pad correctly on time dimension only
    pad_amount = max_len - T
    padding = np.zeros((pad_amount, F), dtype=arr.dtype)

    return np.vstack([arr, padding])


    # pad
    pad_width = max_len - length
    return np.pad(
        arr,
        ((0, pad_width), (0, arr.shape[1])),
        mode="constant"
    )

for i, row in tqdm(df.iterrows(), total=len(df)):
    idx = str(row["index"])
    folder_path = os.path.join(BASE_PATH, idx)

    acoustic_path = os.path.join(folder_path, f"{idx}_acoustic.npy")
    visual_path   = os.path.join(folder_path, f"{idx}_visual.npy")

    if not (os.path.exists(acoustic_path) and os.path.exists(visual_path)):
        continue

    acoustic = np.load(acoustic_path, allow_pickle=True)
    visual   = np.load(visual_path, allow_pickle=True)

    padded_acoustic.append(pad_sequence(acoustic))
    padded_visual.append(pad_sequence(visual))

    labels.append(row["label"])
    folds.append(row["fold"])
    indexes.append(idx)

padded_acoustic = np.array(padded_acoustic, dtype=np.float32)
padded_visual   = np.array(padded_visual, dtype=np.float32)
labels          = np.array(labels)
folds           = np.array(folds)
indexes         = np.array(indexes)

print("Done Padding")
print("Acoustic padded shape:", padded_acoustic.shape)
print("Visual padded shape:", padded_visual.shape)
print("Example acoustic shape:", padded_acoustic[0].shape)
print("Example visual shape:", padded_visual[0].shape)


100%|██████████| 961/961 [00:11<00:00, 84.19it/s] 


Done Padding
Acoustic padded shape: (961, 768, 25)
Visual padded shape: (961, 768, 136)
Example acoustic shape: (768, 25)
Example visual shape: (768, 136)


### Normalize Acoustic & Visual Features

In [8]:
import numpy as np

# STEP 2 — NORMALIZATION


def normalize_features(data):
    """
    data shape: (N, T, F)
    N = samples, T = sequence length, F = features
    Normalizes independently for each feature.
    """
    mean = data.mean(axis=(0,1), keepdims=True)
    std = data.std(axis=(0,1), keepdims=True) + 1e-6
    return (data - mean) / std

acoustic_norm = normalize_features(padded_acoustic)
visual_norm = normalize_features(padded_visual)

print("Acoustic normalized shape:", acoustic_norm.shape)
print("Visual normalized shape:", visual_norm.shape)
print("Example acoustic stats (mean/std):", acoustic_norm.mean(), acoustic_norm.std())
print("Example visual stats (mean/std):", visual_norm.mean(), visual_norm.std())

print("done step 2")


Acoustic normalized shape: (961, 768, 25)
Visual normalized shape: (961, 768, 136)
Example acoustic stats (mean/std): -9.839021e-06 1.0000722
Example visual stats (mean/std): -2.3238474e-06 1.0001274
done step 2


### Split Data + Convert to PyTorch Tensors

In [9]:
import torch
from sklearn.preprocessing import LabelEncoder

# Step 3.1 — Encode labels

le = LabelEncoder()
labels_encoded = le.fit_transform(labels_list)

print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))


# Step 3.2 — Convert lists to arrays

A = np.array(acoustic_norm)   # (961, 512, 25)
V = np.array(visual_norm)     # (961, 512, 136)
y = np.array(labels_encoded)
folds = np.array(folds_list)

print("Shapes:", A.shape, V.shape, y.shape)


# Step 3.3 — Split using your fold column

A_train = A[folds == "train"]
V_train = V[folds == "train"]
y_train = y[folds == "train"]

A_valid = A[folds == "valid"]
V_valid = V[folds == "valid"]
y_valid = y[folds == "valid"]

A_test = A[folds == "test"]
V_test = V[folds == "test"]
y_test = y[folds == "test"]

print("Train:", A_train.shape, V_train.shape, y_train.shape)
print("Valid:", A_valid.shape, V_valid.shape, y_valid.shape)
print("Test:",  A_test.shape,  V_test.shape,  y_test.shape)

# Step 3.4 — Convert to Torch tensors

# Convert numpy to float32 and torch tensors
A_train = A_train.astype(np.float32)
V_train = V_train.astype(np.float32)

A_valid = A_valid.astype(np.float32)
V_valid = V_valid.astype(np.float32)

A_test = A_test.astype(np.float32)
V_test = V_test.astype(np.float32)

A_train_t = torch.from_numpy(A_train)
V_train_t = torch.from_numpy(V_train)
y_train_t = torch.tensor(y_train, dtype=torch.long)

A_valid_t = torch.from_numpy(A_valid)
V_valid_t = torch.from_numpy(V_valid)
y_valid_t = torch.tensor(y_valid, dtype=torch.long)

A_test_t = torch.from_numpy(A_test)
V_test_t = torch.from_numpy(V_test)
y_test_t = torch.tensor(y_test, dtype=torch.long)

print("Tensor shapes ready for dataloaders")


Label mapping: {np.str_('depression'): np.int64(0), np.str_('normal'): np.int64(1)}
Shapes: (961, 768, 25) (961, 768, 136) (961,)
Train: (647, 768, 25) (647, 768, 136) (647,)
Valid: (102, 768, 25) (102, 768, 136) (102,)
Test: (212, 768, 25) (212, 768, 136) (212,)
Tensor shapes ready for dataloaders


### Dataloaders + Masks

In [10]:
# FIXED STEP 5 — Multimodal Dataset

from torch.utils.data import Dataset, DataLoader

# FIXED STEP 5 — Multimodal Dataset

class MultimodalDataset(Dataset):
    def __init__(self, acoustic, visual, labels):
        self.acoustic = acoustic
        self.visual = visual
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.acoustic[idx], self.visual[idx], self.labels[idx]

train_ds = MultimodalDataset(A_train_t, V_train_t, y_train_t)
valid_ds = MultimodalDataset(A_valid_t, V_valid_t, y_valid_t)
test_ds  = MultimodalDataset(A_test_t,  V_test_t,  y_test_t)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=16)
test_loader  = DataLoader(test_ds, batch_size=16)

print("Dataset ready. Each batch returns (a, v, y)")


Dataset ready. Each batch returns (a, v, y)


### Define the MulT (Multimodal Transformer) Architecture

### import libraries

In [11]:
import torch
import torch.nn as nn

import torch.nn.functional as F


### Define the Cross-Modal Attention Block

In [12]:
class CrossModalTransformer(nn.Module):
    def __init__(self, dim_q, dim_kv, hidden_dim=128, num_heads=4):
        super().__init__()

        self.query_proj = nn.Linear(dim_q, hidden_dim)
        self.key_proj   = nn.Linear(dim_kv, hidden_dim)
        self.value_proj = nn.Linear(dim_kv, hidden_dim)

        self.attention = nn.MultiheadAttention(hidden_dim, num_heads)

        self.ff = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, q, kv):
        # project
        Q = self.query_proj(q)
        K = self.key_proj(kv)
        V = self.value_proj(kv)

        # Multi-head attention
        attn_out, _ = self.attention(Q.transpose(0,1),
                                     K.transpose(0,1),
                                     V.transpose(0,1))

        attn_out = attn_out.transpose(0,1)

        # Feedforward
        out = self.ff(attn_out)

        return out


### Create the MulT Model

In [ ]:
# class MulTModel(nn.Module):
#     def __init__(self, audio_dim=25, visual_dim=136, hidden_dim=128, num_heads=4):
#         super().__init__()

#         self.audio_proj  = nn.Linear(audio_dim, hidden_dim)
#         self.visual_proj = nn.Linear(visual_dim, hidden_dim)

#         # Cross-modal attention
#         self.video_to_audio = CrossModalTransformer(dim_q=hidden_dim, dim_kv=hidden_dim,
#                                                     hidden_dim=hidden_dim, num_heads=num_heads)
#         self.audio_to_video = CrossModalTransformer(dim_q=hidden_dim, dim_kv=hidden_dim,
#                                                     hidden_dim=hidden_dim, num_heads=num_heads)

#         # Fusion layer
#         self.fusion = nn.Linear(hidden_dim*2, hidden_dim)

#         # Classification
#         self.classifier = nn.Linear(hidden_dim, 2)

#     def forward(self, audio, visual):
#         # Initial projections
#         A = self.audio_proj(audio)      # (B, 512, H)
#         V = self.visual_proj(visual)    # (B, 512, H)

#         # Cross modal attention
#         A2 = self.video_to_audio(A, V)
#         V2 = self.audio_to_video(V, A)

#         # Global pooling
#         A_pool = A2.mean(dim=1)
#         V_pool = V2.mean(dim=1)

#         fused = torch.cat([A_pool, V_pool], dim=1)

#         fused = self.fusion(fused)

#         out = self.classifier(fused)
#         return out


## 2nd attempt : improved model

In [ ]:
# import torch
# import torch.nn as nn


# # Cross-Modal Transformer Block (Improved)

# class CrossModalTransformer(nn.Module):
#     def __init__(self, hidden_dim=128, num_heads=4, dropout=0.2):
#         super().__init__()

#         self.attn = nn.MultiheadAttention(
#             embed_dim=hidden_dim,
#             num_heads=num_heads,
#             dropout=dropout,
#             batch_first=True
#         )

#         self.norm1 = nn.LayerNorm(hidden_dim)
#         self.norm2 = nn.LayerNorm(hidden_dim)

#         self.ff = nn.Sequential(
#             nn.Linear(hidden_dim, hidden_dim * 4),
#             nn.ReLU(),
#             nn.Dropout(dropout),
#             nn.Linear(hidden_dim * 4, hidden_dim),
#             nn.Dropout(dropout),
#         )

#     def forward(self, q, kv):
#         attn_out, _ = self.attn(q, kv, kv)
#         q = self.norm1(q + attn_out)

#         ff_out = self.ff(q)
#         out = self.norm2(q + ff_out)
#         return out


# # Stronger MulT Fusion Model
# class MulTModel(nn.Module):
#     def __init__(self, audio_dim=25, visual_dim=136, hidden_dim=128, num_heads=4, dropout=0.3):
#         super().__init__()

#         # Projection layers
#         self.audio_proj = nn.Sequential(
#             nn.Linear(audio_dim, hidden_dim),
#             nn.ReLU(),
#             nn.Dropout(dropout),
#             nn.Linear(hidden_dim, hidden_dim),
#             nn.LayerNorm(hidden_dim)
#         )

#         self.visual_proj = nn.Sequential(
#             nn.Linear(visual_dim, hidden_dim),
#             nn.ReLU(),
#             nn.Dropout(dropout),
#             nn.Linear(hidden_dim, hidden_dim),
#             nn.LayerNorm(hidden_dim)
#         )

#         # Cross-modal attention
#         self.video_to_audio = CrossModalTransformer(hidden_dim, num_heads, dropout)
#         self.audio_to_video = CrossModalTransformer(hidden_dim, num_heads, dropout)

#         # Gated fusion
#         self.gate = nn.Sequential(
#             nn.Linear(hidden_dim * 2, hidden_dim),
#             nn.Sigmoid()
#         )

#         # Fusion feedforward layer
#         self.fusion_ff = nn.Sequential(
#             nn.Linear(hidden_dim * 2, hidden_dim),
#             nn.ReLU(),
#             nn.Dropout(dropout),
#             nn.Linear(hidden_dim, hidden_dim),
#             nn.LayerNorm(hidden_dim)
#         )

#         # Classification head (fixed integer sizes)
#         self.classifier = nn.Sequential(
#             nn.Linear(hidden_dim, hidden_dim // 2),
#             nn.ReLU(),
#             nn.Dropout(dropout),
#             nn.Linear(hidden_dim // 2, 2)
#         )

#     def forward(self, audio, visual):
#         A = self.audio_proj(audio)
#         V = self.visual_proj(visual)

#         A2 = self.video_to_audio(A, V)
#         V2 = self.audio_to_video(V, A)

#         A_pool = A2.mean(dim=1)
#         V_pool = V2.mean(dim=1)

#         combined = torch.cat([A_pool, V_pool], dim=1)
#         gate = self.gate(combined)

#         fused = gate * A_pool + (1 - gate) * V_pool
#         fused = self.fusion_ff(torch.cat([A_pool, V_pool], dim=1))

#         out = self.classifier(fused)
#         return out


### 3rd attempt with positional encoding

In [13]:
import torch
import torch.nn as nn
import numpy as np


# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, dim, max_len=768):
        super().__init__()
        pe = torch.zeros(max_len, dim)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dim, 2) * (-np.log(10000.0) / dim))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.pe = pe.unsqueeze(0)  # shape: (1, max_len, dim)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :].to(x.device)


# Cross-Modal Transformer Block
class CrossModalTransformer(nn.Module):
    def __init__(self, hidden_dim=128, num_heads=4, dropout=0.2):
        super().__init__()

        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

        self.ff = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.Dropout(dropout),
        )

    def forward(self, q, kv):
        attn_out, _ = self.attn(q, kv, kv)
        q = self.norm1(q + attn_out)

        ff_out = self.ff(q)
        out = self.norm2(q + ff_out)
        return out


# MulT Fusion Model (with positional encoding)
class MulTModel(nn.Module):
    def __init__(self, audio_dim=25, visual_dim=136, hidden_dim=128, num_heads=4, dropout=0.3):
        super().__init__()

        # Projection layers
        self.audio_proj = nn.Sequential(
            nn.Linear(audio_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim)
        )

        self.visual_proj = nn.Sequential(
            nn.Linear(visual_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim)
        )

        # Positional Encoding
        self.pos_enc = PositionalEncoding(hidden_dim)

        # Cross-modal attention
        self.video_to_audio = CrossModalTransformer(hidden_dim, num_heads, dropout)
        self.audio_to_video = CrossModalTransformer(hidden_dim, num_heads, dropout)

        # Gated fusion
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Sigmoid()
        )

        # Fusion layer
        self.fusion_ff = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim)
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 2)
        )

    def forward(self, audio, visual):
        A = self.audio_proj(audio)
        V = self.visual_proj(visual)

        # ADD POSITIONAL ENCODING
        A = self.pos_enc(A)
        V = self.pos_enc(V)

        # Cross-modal
        A2 = self.video_to_audio(A, V)
        V2 = self.audio_to_video(V, A)

        # Pool
        A_pool = A2.mean(dim=1)
        V_pool = V2.mean(dim=1)

        # Gate
        combined = torch.cat([A_pool, V_pool], dim=1)
        gate = self.gate(combined)

        fused = gate * A_pool + (1 - gate) * V_pool
        fused = self.fusion_ff(torch.cat([A_pool, V_pool], dim=1))

        out = self.classifier(fused)
        return out


### Instantiate model

In [14]:
model = MulTModel(hidden_dim=128)
model = model.to("cuda")


In [15]:

# STEP 6 — TRAIN MULT MODEL

import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=2e-4, weight_decay=1e-5)

# OPTIONAL but recommended: learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3
)

EPOCHS = 30
best_val_acc = 0
patience = 6
wait = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for a, v, y in tqdm(train_loader):
        a, v, y = a.to(device), v.to(device), y.to(device)

        optimizer.zero_grad()
        preds = model(a, v)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # Validation

    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for a, v, y in valid_loader:
            a, v, y = a.to(device), v.to(device), y.to(device)
            preds = model(a, v)

            all_preds.extend(preds.argmax(dim=1).cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    val_acc = (np.array(all_preds) == np.array(all_labels)).mean()
    scheduler.step(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss={total_loss:.4f} | Val Acc={val_acc:.4f}")

    # Early Stopping

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        wait = 0
        torch.save(model.state_dict(), "best_mult.pt")
        print("Saved best model.")
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping triggered.")
            break

print("Training complete.")


100%|██████████| 41/41 [00:02<00:00, 14.41it/s]


Epoch 1/30 | Loss=28.0554 | Val Acc=0.5588
Saved best model.


100%|██████████| 41/41 [00:02<00:00, 18.87it/s]


Epoch 2/30 | Loss=27.1127 | Val Acc=0.6373
Saved best model.


100%|██████████| 41/41 [00:02<00:00, 19.05it/s]


Epoch 3/30 | Loss=25.6446 | Val Acc=0.6765
Saved best model.


... (16 outputs left out)

100%|██████████| 41/41 [00:02<00:00, 19.11it/s]


Epoch 12/30 | Loss=21.1821 | Val Acc=0.6961


100%|██████████| 41/41 [00:02<00:00, 18.80it/s]


Epoch 13/30 | Loss=21.3985 | Val Acc=0.7157
Early stopping triggered.
Training complete.


### evaluate the best saved model on the test set.

In [16]:
# STEP 7 — LOAD BEST MODEL

best_model = MulTModel(
    audio_dim=25,
    visual_dim=136,
    hidden_dim=128,
    num_heads=4
)

best_model.load_state_dict(torch.load("best_mult.pt"))
best_model = best_model.to(device)
best_model.eval()

# STEP 8 — TEST ACCURACY
all_preds = []
all_labels = []

with torch.no_grad():
    for a, v, y in test_loader:
        a, v, y = a.to(device), v.to(device), y.to(device)
        preds = best_model(a, v)

        all_preds.extend(preds.argmax(dim=1).cpu().numpy())
        all_labels.extend(y.cpu().numpy())

test_acc = (np.array(all_preds) == np.array(all_labels)).mean()
print("Test Accuracy:", test_acc)


Test Accuracy: 0.7264150943396226
